In [5]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [6]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [7]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [8]:
dataset = generate_dataset()
print(dataset)

[{'task': 'Write a Python function that parses an AWS CloudFormation template (JSON string) and returns a list of all EC2 instance logical IDs defined in the Resources section.'}, {'task': "Create a JSON policy document for an AWS IAM role that allows a Lambda function to read objects from a specific S3 bucket named 'my-data-bucket' and write CloudWatch logs."}, {'task': 'Write a regular expression that matches and extracts the AWS region from an S3 bucket ARN in the format: arn:aws:s3:::bucket-name or arn:aws:s3:region:account-id:bucket/bucket-name'}]


In [9]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [10]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [11]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [12]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [13]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [16]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS CloudFormation EC2 Instance Parser\n\nHere's a Python function that parses a CloudFormation template and extracts EC2 instance logical IDs:\n\n```python\nimport json\nfrom typing import List\n\ndef get_ec2_instance_ids(template_json: str) -> List[str]:\n    \"\"\"\n    Parse an AWS CloudFormation template and return a list of all EC2 instance logical IDs.\n    \n    Args:\n        template_json: A JSON string representing the CloudFormation template\n        \n    Returns:\n        A list of logical IDs for EC2 instances defined in the Resources section\n        \n    Raises:\n        json.JSONDecodeError: If the template_json is not valid JSON\n        KeyError: If the template doesn't have a Resources section\n    \"\"\"\n    template = json.loads(template_json)\n    \n    ec2_instance_ids = []\n    \n    # Get the Resources section\n    resources = template.get('Resources', {})\n    \n    # Iterate through resources and find EC2 instances\n    for logical_